# U7-04 · Una Deep Q-Network en CartPole

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 7, apartados 7.7 y 7.8*

Ejecuta cada celda con **Mayús + Intro**. La red está escrita con NumPy para ver todo lo que hace. El entrenamiento tarda unos minutos en el navegador.

In [ ]:
import sys, types
if sys.platform == "emscripten":
    # En el navegador (Pyodide) no existe este módulo de procesos
    # paralelos; Gymnasium lo importa aunque aquí no lo use.
    m = types.ModuleType("_multiprocessing")
    m.SemLock = type("SemLock", (), {"SEM_VALUE_MAX": 2**31 - 1})
    m.sem_unlink, m.flags = None, {}
    sys.modules["_multiprocessing"] = m
%pip install -q gymnasium
import gymnasium
print("Gymnasium", gymnasium.__version__)

## 1. La red y el algoritmo

Una red densa 4-64-64-2 con ReLU, pérdida de Huber y optimizador Adam. La DQN usa repetición de experiencia y red objetivo; cada 20 episodios se evalúa y se guarda la mejor red.

In [ ]:
import numpy as np, gymnasium as gym

ESCALA = np.array([2.4, 3.0, 0.21, 3.0])   # rangos típicos del estado

class Red:
    """Red densa 4-64-64-2 con ReLU, entrenada con Adam (NumPy)."""
    def __init__(self, tamanos, rng):
        self.W = [rng.normal(0, np.sqrt(2 / a), (a, b))
                  for a, b in zip(tamanos[:-1], tamanos[1:])]
        self.b = [np.zeros(b) for b in tamanos[1:]]
        self.m = [np.zeros_like(p) for p in self.W + self.b]
        self.v = [np.zeros_like(p) for p in self.W + self.b]
        self.t = 0

    def predecir(self, x):
        x = x / ESCALA
        self.capas = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = x @ W + b
            if i < len(self.W) - 1:
                x = np.maximum(x, 0)
            self.capas.append(x)
        return x

    def ajustar(self, x, acciones, objetivo, lr=1e-3):
        q = self.predecir(x)
        err = np.zeros_like(q)
        idx = np.arange(len(x))
        dif = q[idx, acciones] - objetivo
        err[idx, acciones] = np.clip(dif, -1, 1) / len(x)   # pérdida de Huber
        grads_W, grads_b = [], []
        for i in reversed(range(len(self.W))):
            grads_W.insert(0, self.capas[i].T @ err)
            grads_b.insert(0, err.sum(0))
            if i > 0:
                err = (err @ self.W[i].T) * (self.capas[i] > 0)
        self.t += 1
        params = self.W + self.b
        for k, (p, g) in enumerate(zip(params, grads_W + grads_b)):
            self.m[k] = 0.9 * self.m[k] + 0.1 * g
            self.v[k] = 0.999 * self.v[k] + 0.001 * g * g
            mh = self.m[k] / (1 - 0.9 ** self.t)
            vh = self.v[k] / (1 - 0.999 ** self.t)
            p -= lr * mh / (np.sqrt(vh) + 1e-8)

    def copiar_de(self, otra):
        self.W = [w.copy() for w in otra.W]
        self.b = [b.copy() for b in otra.b]


def entrenar_dqn(episodios=600, semilla=0, gamma=0.99, lote=64,
                 memoria_max=50_000, copiar_cada=500, verbose=False,
                 lr=5e-4, decaer=10_000):
    rng = np.random.default_rng(semilla)
    env = gym.make("CartPole-v1")
    red = Red([4, 64, 64, 2], rng)
    objetivo = Red([4, 64, 64, 2], rng)
    objetivo.copiar_de(red)
    mejor = Red([4, 64, 64, 2], rng)
    mejor.copiar_de(red)
    mejor_nota = -1
    memoria = []
    pasos, retornos = 0, []
    for ep in range(episodios):
        s, _ = env.reset(seed=semilla * 10_000 + ep)
        fin, G = False, 0
        while not fin:
            eps = max(0.05, 1 - pasos / decaer)
            if rng.random() < eps:
                a = int(rng.integers(2))
            else:
                a = int(np.argmax(red.predecir(s[None])[0]))
            s2, r, term, trunc, _ = env.step(a)
            fin = term or trunc
            memoria.append((s, a, r, s2, term))
            if len(memoria) > memoria_max:
                memoria.pop(0)
            s, G, pasos = s2, G + r, pasos + 1
            if len(memoria) >= 1000:
                muestra = [memoria[i] for i in rng.integers(len(memoria), size=lote)]
                S, A, R, S2, D = map(np.array, zip(*muestra))
                q_sig = objetivo.predecir(S2).max(1)
                y = R + gamma * q_sig * (1 - D)
                red.ajustar(S, A, y, lr)
            if pasos % copiar_cada == 0:
                objetivo.copiar_de(red)
        retornos.append(G)
        if (ep + 1) % 20 == 0:                 # cada 20 episodios, se evalúa
            nota = evaluar(red, n=5, semilla=50_000 + ep).mean()
            if nota >= mejor_nota:             # y se guarda la mejor red
                mejor_nota = nota
                mejor.copiar_de(red)
            if verbose:
                print(ep + 1, pasos, round(np.mean(retornos[-20:]), 1), nota)
            if mejor_nota == 500:
                break
    return mejor, np.array(retornos)


def evaluar(red, n=20, semilla=999):
    env = gym.make("CartPole-v1")
    res = []
    for i in range(n):
        s, _ = env.reset(seed=semilla + i)
        fin, G = False, 0
        while not fin:
            s, r, t1, t2, _ = env.step(int(np.argmax(red.predecir(s[None])[0])))
            fin, G = t1 or t2, G + r
        res.append(G)
    return np.array(res)

## 2. Entrenar (ejemplo 6)

Con la semilla 1, la mejor red alcanza 500 en la evaluación final del libro. Si tienes prisa, prueba con `episodios=200`: verás que aún no ha aprendido.

In [ ]:
import time
t0 = time.time()
red, retornos = entrenar_dqn(semilla=1)
print(f"{len(retornos)} episodios en {time.time() - t0:.0f} s")
print("Evaluación en 20 episodios nuevos:", evaluar(red, n=20).mean())

In [ ]:
import matplotlib.pyplot as plt
plt.plot(retornos, alpha=0.4, label="episodio")
plt.plot(np.convolve(retornos, np.ones(20) / 20, mode="valid"), label="media de 20")
plt.axhline(500, ls=":", color="gray"); plt.xlabel("episodio"); plt.ylabel("retorno")
plt.legend(); plt.show()

**Prueba.** Cambia `copiar_cada=500` por `copiar_cada=1` (es decir, sin red objetivo) y compara la curva. ¿Qué ocurre con la estabilidad?

> Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518, 529–533. https://doi.org/10.1038/nature14236
>
> Lin, L.-J. (1992). Self-improving reactive agents based on reinforcement learning, planning and teaching. *Machine Learning*, 8, 293–321. https://doi.org/10.1007/BF00992699